# `intro.ipynb`


## langchain
`uv add langchain_openai langchain_core langchain_community`

## 한줄요약
- langchain == Agent Builder 다.

- langsmith ->  디버깅을 담당해줌

In [1]:
# 환경변수(.env) 내용을 로드
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Structured Output 지정하기
'''
{
    'user_input' : 원본 사용자 질문
    'answer' : AI가 작성한 답변
}
'''
from pydantic import BaseModel, Field

class AgentResponse(BaseModel):
    user_input: str = Field(description="원본 사용자 질문")
    answer: str = Field(description="AI가 작성한 답변")

In [5]:
from langchain.agents import create_agent
#메모리 추가
from langgraph.checkpoint.memory import InMemorySaver

# Agent에서 쓸 tool
def get_weather(city : str):# 매개변수의 데이터타입 지정해주는것도 AI입장에서 큰 도움
  '''주어진 도시의 날씨를 확인하는 Tool''' ## 독스크립션 n8n에서 agent tool의 description 역할을 함
  return f'{city}는 화창합니다.'

def calculator(num1:float, num2: float, opr : str):
  '''opr을 보고 num1과 num2의 사칙연산을 진행하는 툴
     opr에는 '-', '+' 기호만 넣을 수 있음
  '''
  result = 0
  if opr == '+':
    result = num1 + num2
  elif opr == '-':
    result = num1 - num2
  else:
    return ''-', '+' 기호만 이용 할 수 있습니다'

  return result

memory = InMemorySaver()

agent = create_agent(
  model='openai:gpt-4.1-mini',
  tools=[get_weather, calculator],
  system_prompt="You are a helpful assistant. Answer in KOR",
  checkpointer = memory, #메모리
  # response_format = AgentResponse() , #Structured OutPut
)

In [6]:
# 메모리에서 대화 내역을 구분할 key가 필요(n8n에서의 session key)
thread_config = {
    'configurable':{'thread_id':'1234'} # 세션 번호
}

# 에이전트 실행
agent.invoke(
  # 1번인자 : 메세지
  {
      'messages':[
      {'role':'user', 'content':'아까 실행 결과 얼마였지??'}
    ]
  },
  # 2번인자 : 설정
  thread_config,
)

{'messages': [HumanMessage(content='아까 실행 결과 얼마였지??', additional_kwargs={}, response_metadata={}, id='559d9bbc-fc47-476f-a6a1-2ec75843afd6'),
  AIMessage(content='아까 실행 결과가 무엇인지 구체적으로 알려주시면 더 정확하게 도와드릴 수 있습니다. 어떤 계산이나 작업 결과를 말씀하시는 건가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 123, 'total_tokens': 160, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6fef50c0ab', 'id': 'chatcmpl-EJGRKiqyOLN1Cwd6pvPa4lvQ6OnuC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a05c9c-29d9-7410-9459-df7d030b2c60-0', tool_calls=[], invali

In [7]:
user_input = input()

thread_config = {
    'configurable':{'thread_id':'1234'} # 세션 번호
}

while user_input != '종료':
  print('사용자:', user_input)
  result = agent.invoke(
    # 1번인자 : 메세지
    {
        'messages':[
        {'role':'user', 'content':user_input}
      ]
    },
    # 2번인자 : 설정
    thread_config,
)
  print('AI:', result['messages'][-1].content)
  user_input = input()

사용자: 
AI: 아까 실행 결과가 무엇인지 다시 한 번 말씀해 주시겠어요? 그래야 정확히 알려드릴 수 있습니다.


## 실습예제

- 로또 Agent 만들기
- Tool
    - 함수이름: `get_lotto_info`
    - `requests` 로 요청 보내는 도구
    - url : `https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do?srchStrLtEpsd=1232&srchEndLtEpsd=1232`
    - `srchStrLtEpsd` : 조회 시작 회차
    - `srchEndLtEpsd` : 조회 종료 회차
    - 배개변수: `start_round: int, end_round: int`(각각 조회할 때 시작회차와 마지막 회차의 번호)
    - 설명(description): LLM이 잘 사용 할 수 있도록 작성
- Memory
    - `InMemorySaver`
    - `thread_id` 는 편한대로
- Model
    - `openai:gpt-4.1-mini`
- System prompt
    - 잘 동작하도록 작성
    - 오늘 날짜 + 시간 넣기 (검색 필요)
    - 오늘 날짜 기준 최신회차는 1239
- 대화형 UI로 만들기
- 예시 대화
    - Q : 가장 최근회차의 1등 당첨금액은 얼마야? -> A: 22.1억
    - Q : 저번 회차는 몇명이 1등 당첨됐어? -> 


In [ ]:
import os
import requests
from dotenv import load_dotenv
from datetime import datetime
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

lotto_memory = InMemorySaver()
now = datetime.today()

# 로또 불러오기 툴
def get_lotto_info(start_round:int, end_round:int):
    '''입력 범위의 로또 정보를 불러오는 툴이야.
    start_round에 들어갈 내용은 조회 시작회차, end_round 는 조회 종료 회차야
    '''
    URL = f'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do?srchStrLtEpsd={start_round}&srchEndLtEpsd={end_round}'
    res = requests.get(URL)
    parsing_data = res.json()
    return parsing_data

lotto_agent = create_agent(
  model='openai:gpt-4.1-mini',
  tools=[get_lotto_info],
  system_prompt= f'너는 로또의 정보에 대해서 말해주는 에이전트야 오늘 날짜는 {now} 이고 가장 최근 회차는 1239 회야',
  checkpointer = lotto_memory, #메모리
)


# 메모리에서 대화 내역을 구분할 key가 필요(n8n에서의 session key)
thread_config = {
    'configurable':{'thread_id':'lotto_man'} # 세션 번호
}

# 에이전트 실행
lotto_agent.invoke(
  # 1번인자 : 메세지
  {
      'messages':[
      {'role':'user', 'content':'최근회차 1등 당첨금액 얼마야'}
    ]
  },
  # 2번인자 : 설정
  thread_config,
)
